## Overview
Build a conversational RAG agent using **LangGraph** with **multiple tools** and **memory**.

- **LangChain/LangGraph agents**
- **Multiple tools** 
- **Memory** 
- **LangSmith monitoring** 
- RAG with citations
- Timestamp-aware responses

## Agent Tools:
1. **Retrieval Tool** - Search video transcripts
2. **Video Metadata Tool** - Get video details
3. **Source Citation Tool** - Format sources with timestamps
4. **Video Search Tool** - Find videos by topic

## Output:
- Fully functional RAG agent
- Ready for Gradio deployment

## 1. Install & Import Dependencies

In [1]:
# Run if needed
# !pip install langchain langchain-openai langchain-pinecone langgraph langsmith python-dotenv rank-bm25

In [6]:
import os
import json
from typing import TypedDict, List, Dict, Any, Annotated
from datetime import datetime

from dotenv import load_dotenv

# LangChain Core
from langchain.schema import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.tools import Tool, StructuredTool
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferMemory
from langchain.agents import create_openai_functions_agent, AgentExecutor

# LangGraph (REQUIRED by Carlos for agents)


from langgraph.checkpoint.memory import MemorySaver


# Pinecone
from pinecone import Pinecone


print("✅ All libraries imported")

✅ All libraries imported


In [7]:
import importlib.metadata
print(importlib.metadata.version("langgraph"))

0.2.16


## 2. Configuration

In [9]:
# Load environment variables
load_dotenv()

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

# LangSmith Configuration
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "youtube-rag-engineering-chatbot"
os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY or ""

# Pinecone Configuration
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "youtube-rag-mechanical-engineering")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "efficient-engineer-v3")

# Model Configuration  
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o")

# Retrieval Configuration
TOP_K = 5  


print("✅ Configuration loaded")
print(f"   Chat Model: {CHAT_MODEL}")
print(f"   Embedding Model: {EMBEDDING_MODEL}")
print(f"   Pinecone Index: {INDEX_NAME}")
print(f"   Namespace: {NAMESPACE}")
print(f"   LangSmith Tracing: Enabled")
print(f"   Advanced Retrieval: Top K={TOP_K}")

✅ Configuration loaded
   Chat Model: gpt-4o
   Embedding Model: text-embedding-3-small
   Pinecone Index: youtube-rag-mechanical-engineering
   Namespace: efficient-engineer-v3
   LangSmith Tracing: Enabled
   Advanced Retrieval: Top K=5


## 4. Initialize LangChain Components

In [10]:
# Initialize LLM
llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0,
    openai_api_key=OPENAI_API_KEY,
)

# Initialize Embeddings
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=OPENAI_API_KEY,
)

# Initialize Pinecone VectorStore
vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    namespace=NAMESPACE,
)

print("✅ LLM initialized")
print("✅ Embeddings initialized")
print("✅ VectorStore connected")

✅ LLM initialized
✅ Embeddings initialized
✅ VectorStore connected


## 5. Define Agent Tools (REQUIRED by Carlos)

Carlos requires "agents with **several tools**". Here we define 4 tools.

In [11]:
# ============================================================================
# TOOL 1: Retrieval Tool
# ============================================================================
def search_video_transcripts(query: str) -> str:
    """Search YouTube video transcripts for relevant information.
    
    Args:
        query: The search query
        k: Number of results to return (default: 5)
    
    Returns:
        Formatted search results with timestamps and metadata
    """
    try:
        print(f"\n🔍 Tool called with query: '{query}'")

        # Use LangChain similarity search
        results = vectorstore.similarity_search_with_score(
            query=query,
            k=TOP_K,
            namespace=NAMESPACE,
        )
        
        if not results:
            return "No relevant information found in video transcript database."
        
        # Format results
        output = []
        for i, (doc, score) in enumerate(results, 1):
            meta = doc.metadata
            output.append(
                f"[Source {i}] (Score: {score:.3f})\n"
                f"Video: {meta.get('video_title', 'Unknown')}\n"
                f"Channel: {meta.get('channel', 'Unknown')}\n"
                f"Timestamp: {meta.get('start_time_str')} - {meta.get('end_time_str')}\n"
                f"URL: {meta.get('video_url', '')}?t={int(meta.get('start_time', 0))}s\n"
                f"Content: {doc.page_content[:300]}...\n"
            )
        
        return "\n".join(output)
    
    except Exception as e:
        error_msg = f"Error searching transcripts: {str(e)}"
        print(f"❌ {error_msg}")
        return error_msg


# ============================================================================
# TOOL 2: Video Metadata Tool
# ============================================================================
def get_video_metadata(video_id: str) -> str:
    """Get metadata for a specific video.
    
    Args:
        video_id: YouTube video ID
    
    Returns:
        Video metadata including title, channel, duration, etc.
    """
    try:
        # Search for any chunk from this video
        results = vectorstore.similarity_search(
            query="",  # Empty query
            k=1,
            namespace=NAMESPACE,
            filter={"video_id": video_id},
        )
        
        if not results:
            return f"No metadata found for video ID: {video_id}"
        
        meta = results[0].metadata
        return (
            f"Video Title: {meta.get('video_title', 'Unknown')}\n"
            f"Channel: {meta.get('channel', 'Unknown')}\n"
            f"Video ID: {video_id}\n"
            f"URL: {meta.get('video_url', 'N/A')}\n"
            f"Upload Date: {meta.get('upload_date', 'Unknown')}\n"
            f"Total Chunks: {meta.get('total_chunks', 'Unknown')}\n"
            f"Language: {meta.get('transcript_language', 'Unknown')}"
        )
    
    except Exception as e:
        return f"Error getting metadata: {str(e)}"


# ============================================================================
# TOOL 3: Format Source Citations
# ============================================================================
def format_sources(source_numbers: List[int]) -> str:
    """Format source citations with clickable timestamp links.
    
    Args:
        source_numbers: List of source numbers to format (e.g., [1, 2, 3])
    
    Returns:
        Formatted citation text
    """
    if not source_numbers:
        return "No sources to format."
    
    citations = [f"[Source {num}]" for num in source_numbers]
    return "Sources: " + ", ".join(citations)


# ============================================================================
# TOOL 4: Search Videos by Topic
# ============================================================================
def search_videos_by_topic(topic: str) -> str:
    """Find videos that discuss a specific topic.
    
    Args:
        topic: Topic to search for
    
    Returns:
        List of relevant videos
    """
    max_videos = 5

    try:
        results = vectorstore.similarity_search(
            query=topic,
            k=max_videos * 3,
            namespace=NAMESPACE,
        )

        seen_videos = set()
        videos = []

        for doc in results:
            video_id = doc.metadata.get("video_id")
            if video_id and video_id not in seen_videos:
                seen_videos.add(video_id)
                videos.append({
                    "title": doc.metadata.get("video_title", "Unknown"),
                    "url": doc.metadata.get("video_url", "N/A"),
                    "channel": doc.metadata.get("channel", "Unknown"),
                })

            if len(videos) >= max_videos:
                break

        if not videos:
            return f"No videos found about '{topic}'."

        output = [f"Found {len(videos)} videos about '{topic}':\n"]
        for i, video in enumerate(videos, 1):
            output.append(
                f"{i}. {video['title']}\n"
                f"   Channel: {video['channel']}\n"
                f"   URL: {video['url']}"
            )

        return "\n".join(output)

    except Exception as e:
        return f"Error searching videos: {str(e)}"


# ============================================================================
# Create LangChain Tools (REQUIRED format)
# ============================================================================
tools = [
    Tool(
        name="search_transcripts",
        func=search_video_transcripts,
        description="Search YouTube video transcripts for information. Use this when the user asks questions about the video content. Returns relevant chunks with timestamps and citations.",
    ),
    Tool(
        name="get_video_info",
        func=get_video_metadata,
        description="Get detailed metadata about a specific video by its ID. Use this when you need information about a particular video's properties.",
    ),
    Tool(
        name="format_citations",
        func=format_sources,
        description="Format source citations with numbers. Use this to create proper citations for your answers.",
    ),
    Tool(
        name="find_videos",
        func=search_videos_by_topic,
        description="Find videos that discuss a specific topic or subject. Use this when the user wants to know which videos cover certain topics.",
    ),
]

print(f"✅ Created {len(tools)} agent tools:")
for tool in tools:
    print(f"   - {tool.name}: {tool.description[:60]}...")

✅ Created 4 agent tools:
   - search_transcripts: Search YouTube video transcripts for information. Use this w...
   - get_video_info: Get detailed metadata about a specific video by its ID. Use ...
   - format_citations: Format source citations with numbers. Use this to create pro...
   - find_videos: Find videos that discuss a specific topic or subject. Use th...


## TEST TOOLS DIRECTLY (Before Agent)

In [12]:
# ============================================================================
# CRITICAL: Test tools work BEFORE creating agent
# ============================================================================
print("\n🧪 Testing tool directly...\n")

test_result = search_video_transcripts("what is carbon fiber?")
print(f"\n📊 Tool returned {len(test_result)} characters")

if len(test_result) > 200:
    print("✅ Tool works! Preview:")
    print(test_result[:400] + "...\n")
else:
    print("❌ Tool failed or returned minimal data!")
    print(f"Result: {test_result}\n")
    print("⚠️  Check your Pinecone namespace and data!")


🧪 Testing tool directly...


🔍 Tool called with query: 'what is carbon fiber?'

📊 Tool returned 2484 characters
✅ Tool works! Preview:
[Source 1] (Score: 0.588)
Video: Carbon Fiber - The Material Of The Future?
Channel: Real Engineering
Timestamp: 04:28 - 04:58
URL: https://www.youtube.com/watch?v=aKpDyfJnxQQ?t=268s
Content: . But he was breaking the material across the fibers; if he had attempted to break it along the fibers, he would've had serious difficulty. You see, carbon fiber is made from these tiny thin fibers. This is a...



## Create LangChain Tools - STRONG DESCRIPTIONS

In [13]:
tools = [
    StructuredTool.from_function(
        func=search_video_transcripts,
        name="search_transcripts",
        description="""Search YouTube video transcripts for information about mechanical engineering.

USE THIS TOOL FOR EVERY QUESTION about:
- Engineering concepts (stress, strain, materials, Young's modulus, etc.)
- How mechanical systems work
- Technical explanations
- Material properties
- Design principles
- Any engineering topic

This tool searches actual video transcripts and returns content with timestamps.
Input: A query string (e.g., "stress strain", "Young's modulus", "material failure")
Output: Video content with sources and timestamps

REQUIRED for answering engineering questions!""",
    ),
    StructuredTool.from_function(
        func=search_videos_by_topic,
        name="find_videos",
        description="""Find videos that discuss a specific topic.

Use when user asks: "Which videos discuss X?" or "Show me videos about Y"
Input: A topic string
Output: List of videos with titles and URLs""",
    ),
    StructuredTool.from_function(
        func=get_video_metadata,
        name="get_video_info",
        description="""Get metadata about a specific video.

Use when you have a video_id and need details.
Input: video_id (11-character YouTube ID)
Output: Video title, channel, URL, upload date""",
    ),
]

print(f"✅ Created {len(tools)} LangChain tools")
for tool in tools:
    print(f"   - {tool.name}")

✅ Created 3 LangChain tools
   - search_transcripts
   - find_videos
   - get_video_info


## 6. Create LangGraph Agent (REQUIRED by Carlos)

Using **LangGraph** to build the agent with tools and memory.

In [14]:
# ============================================================================
# FIXED SYSTEM PROMPT - ENFORCES TOOL USAGE
# ============================================================================
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an AI assistant specialized in mechanical engineering YouTube videos.

⚠️ CRITICAL RULES - FOLLOW EXACTLY:

1. You MUST use the search_transcripts tool for EVERY engineering question - NO EXCEPTIONS
2. You can ONLY answer based on the retrieved video content - NEVER use your general training knowledge
3. ALWAYS search first, read the results, then answer based ONLY on what you found
4. If search returns no results, say: "I couldn't find information about that in the videos"
5. ALWAYS cite sources with video titles and timestamps

YOUR PROCESS:
Step 1: User asks question
Step 2: Call search_transcripts with the question
Step 3: Read the returned video content carefully
Step 4: Answer ONLY based on what the tool returned
Step 5: Include source citations (video title, timestamp, URL)

EXAMPLE:
User: "What is stress?"
You: [Call search_transcripts("stress engineering")]
[Read results]
"According to 'Introduction to Stress' at 02:15-03:30, stress is force per unit area... [Source 1]"

WRONG EXAMPLE (DO NOT DO THIS):
User: "What is stress?"
You: "Stress is a physical response..." ❌ NO! This is from your training, not the videos!

Available tools:
- search_transcripts: Search video database (USE FIRST!)
- find_videos: Find videos by topic
- get_video_info: Get video metadata

Remember: You're a VIDEO SEARCH EXPERT, not a general knowledge assistant!
If you don't search the videos, you're failing your purpose!
"""),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Create agent
agent = create_openai_functions_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

# Create memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
)

# Create agent executor - WITH IMPROVEMENTS
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,  # See tool calls
    handle_parsing_errors=True,
    max_iterations=5,  # Allow multiple tool calls
    early_stopping_method="generate",
    return_intermediate_steps=True,  # See what tools were used
)

print("\n✅ LangChain Agent created successfully!")
print("✅ Tool usage ENFORCED in system prompt")
print("✅ Memory enabled")
print("✅ LangSmith tracing active")


✅ LangChain Agent created successfully!
✅ Tool usage ENFORCED in system prompt
✅ Memory enabled
✅ LangSmith tracing active


## 7. Test Agent with Sample Queries

In [16]:
def ask_agent(question: str) -> str:
    """Ask a question to the agent."""
    try:
        response = agent_executor.invoke({
            "input": question,
        })
        return response["output"]
    except Exception as e:
        return f"Error: {str(e)}"


# Test Query 1: Simple question
print("\n" + "="*80)
print("TEST 1: Simple Question")
print("="*80)

question1 = "What is turbulent flow?"
print(f"\nQ: {question1}\n")
answer1 = ask_agent(question1)
print(f"A: {answer1}\n")


TEST 1: Simple Question

Q: What is turbulent flow?



> Entering new AgentExecutor chain...

Invoking: `search_transcripts` with `{'query': 'turbulent flow'}`



🔍 Tool called with query: 'turbulent flow'
[Source 1] (Score: 0.650)
Video: Understanding Laminar and Turbulent Flow
Channel: The Efficient Engineer
Timestamp: 00:39 - 01:26
URL: https://www.youtube.com/watch?v=9A-uUG0WR0w?t=39s
Content: . This is the start of the transition between the laminar and turbulent regimes. If we continue increasing the velocity we end up with fully turbulent flow. Turbulent flow is characterised by chaotic movement and contains swirling regions called eddies. The chaotic motion and eddies result in signif...

[Source 2] (Score: 0.628)
Video: Understanding Laminar and Turbulent Flow
Channel: The Efficient Engineer
Timestamp: 01:05 - 01:59
URL: https://www.youtube.com/watch?v=9A-uUG0WR0w?t=65s
Content: . For turbulent flow we’ll get data that looks like this. This flow is much more complicated. We c

c:\Users\yogan\Documents\JupyterNotebook_ironhack\Bootcamp\week_08_03_final_project\Final-Project---Youtube-Engineering-Chatbot\venv\Lib\site-packages\langchain\memory\chat_memory.py:38: UserWarning: 'ConversationBufferMemory' got multiple output keys: dict_keys(['output', 'intermediate_steps']). The default 'output' key is being used. If this is not desired, please manually set 'output_key'.
  warnings.warn(


In [17]:
# Test Query 2: Follow-up question (tests memory)
print("\n" + "="*80)
print("TEST 2: Follow-up Question (Memory Test)")
print("="*80)

question2 = "Can you explain that in simpler terms?"
print(f"\nQ: {question2}\n")
answer2 = ask_agent(question2)
print(f"A: {answer2}\n")


TEST 2: Follow-up Question (Memory Test)

Q: Can you explain that in simpler terms?



> Entering new AgentExecutor chain...

Invoking: `search_transcripts` with `{'query': 'turbulent flow simple explanation'}`



🔍 Tool called with query: 'turbulent flow simple explanation'
[Source 1] (Score: 0.645)
Video: Understanding Laminar and Turbulent Flow
Channel: The Efficient Engineer
Timestamp: 00:39 - 01:26
URL: https://www.youtube.com/watch?v=9A-uUG0WR0w?t=39s
Content: . This is the start of the transition between the laminar and turbulent regimes. If we continue increasing the velocity we end up with fully turbulent flow. Turbulent flow is characterised by chaotic movement and contains swirling regions called eddies. The chaotic motion and eddies result in signif...

[Source 2] (Score: 0.626)
Video: Understanding Laminar and Turbulent Flow
Channel: The Efficient Engineer
Timestamp: 01:05 - 01:59
URL: https://www.youtube.com/watch?v=9A-uUG0WR0w?t=65s
Content: . For turbulent flow we’ll g

c:\Users\yogan\Documents\JupyterNotebook_ironhack\Bootcamp\week_08_03_final_project\Final-Project---Youtube-Engineering-Chatbot\venv\Lib\site-packages\langchain\memory\chat_memory.py:38: UserWarning: 'ConversationBufferMemory' got multiple output keys: dict_keys(['output', 'intermediate_steps']). The default 'output' key is being used. If this is not desired, please manually set 'output_key'.
  warnings.warn(


In [19]:
# Test Query 3: Video search
print("\n" + "="*80)
print("TEST 3: Video Search")
print("="*80)

question3 = "Which videos discuss turbulent flow?"
print(f"\nQ: {question3}\n")
answer3 = ask_agent(question3)
print(f"A: {answer3}\n")


TEST 3: Video Search

Q: Which videos discuss turbulent flow?



> Entering new AgentExecutor chain...

Invoking: `find_videos` with `{'topic': 'turbulent flow'}`


Found 3 videos about 'turbulent flow':

1. Understanding Laminar and Turbulent Flow
   Channel: The Efficient Engineer
   URL: https://www.youtube.com/watch?v=9A-uUG0WR0w
2. Understanding Aerodynamic Drag
   Channel: The Efficient Engineer
   URL: https://www.youtube.com/watch?v=GMmNKUlXXDs
3. Understanding Viscosity
   Channel: The Efficient Engineer
   URL: https://www.youtube.com/watch?v=VvDJyhYSJv8Here are some videos that discuss turbulent flow:

1. [Understanding Laminar and Turbulent Flow](https://www.youtube.com/watch?v=9A-uUG0WR0w) by The Efficient Engineer
2. [Understanding Aerodynamic Drag](https://www.youtube.com/watch?v=GMmNKUlXXDs) by The Efficient Engineer
3. [Understanding Viscosity](https://www.youtube.com/watch?v=VvDJyhYSJv8) by The Efficient Engineer

These videos cover various aspects of turbulent flow 

c:\Users\yogan\Documents\JupyterNotebook_ironhack\Bootcamp\week_08_03_final_project\Final-Project---Youtube-Engineering-Chatbot\venv\Lib\site-packages\langchain\memory\chat_memory.py:38: UserWarning: 'ConversationBufferMemory' got multiple output keys: dict_keys(['output', 'intermediate_steps']). The default 'output' key is being used. If this is not desired, please manually set 'output_key'.
  warnings.warn(


In [23]:
from datetime import datetime
from langchain.memory import ConversationBufferMemory
from langchain.agents import AgentExecutor, create_tool_calling_agent


def _build_contextual_input(user_input: str, session_history):
    """
    Rewrite vague follow-up prompts into explicit context.
    """
    follow_up_markers = [
        "previous question",
        "previous answer",
        "explain that",
        "explain it",
        "explain the previous question",
        "make it simple",
        "simplify that",
        "simpler manner",
        "simple manner",
        "summarize that",
        "what do you mean",
        "say that simply",
        "explain in simple terms",
    ]

    msg_lower = user_input.lower()

    if session_history and any(marker in msg_lower for marker in follow_up_markers):
        last_user = session_history[-1]["user"]
        last_assistant = session_history[-1]["assistant"]

        return (
            "This is a follow-up request in an ongoing conversation.\n\n"
            f"Previous user question: {last_user}\n"
            f"Previous assistant answer: {last_assistant}\n\n"
            f"Current user request: {user_input}"
        )

    return user_input


def create_chat_session_agent():
    """
    Create a completely fresh memory + agent executor for one chat session.
    """
    session_memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )

    # Reuse your existing llm, tools, and prompt variables
    session_agent = create_tool_calling_agent(
        llm=llm,
        tools=tools,
        prompt=prompt
    )

    session_agent_executor = AgentExecutor(
        agent=session_agent,
        tools=tools,
        memory=session_memory,
        verbose=True,
        handle_parsing_errors=True
    )

    return session_agent_executor, session_memory


def ask_agent(question: str, agent_executor, session_history=None) -> str:
    """
    Ask the session-specific agent a question.
    """
    try:
        contextual_question = _build_contextual_input(question, session_history or [])

        response = agent_executor.invoke({
            "input": contextual_question,
            "current_date": datetime.now().strftime("%Y-%m-%d"),
        })

        return response["output"]

    except Exception as e:
        return f"Error: {str(e)}"


def chat_session():
    """
    Start an interactive chat session with isolated memory.
    Every run gets a fresh executor and fresh memory.
    """
    print("\n" + "=" * 80)
    print("YOUTUBE RAG CHATBOT - Interactive Session")
    print("=" * 80)
    print("Type 'quit', 'exit', or 'q' to end the session.\n")

    agent_executor, session_memory = create_chat_session_agent()
    session_history = []

    while True:
        try:
            user_input = input("You: ").strip()

            if user_input.lower() in ["quit", "exit", "q", ""]:
                print("\nGoodbye! 👋")
                break

            print("\nAssistant: ", end="")
            response = ask_agent(
                question=user_input,
                agent_executor=agent_executor,
                session_history=session_history
            )
            print(response)
            print()

            session_history.append({
                "user": user_input,
                "assistant": response
            })

        except KeyboardInterrupt:
            print("\n\nSession interrupted. Goodbye! 👋")
            break
        except Exception as e:
            print(f"\nError: {e}\n")


# Start interactive session
chat_session()


YOUTUBE RAG CHATBOT - Interactive Session
Type 'quit', 'exit', or 'q' to end the session.


Assistant: 

> Entering new AgentExecutor chain...

Invoking: `search_transcripts` with `{'query': 'turbulence'}`



🔍 Tool called with query: 'turbulence'
[Source 1] (Score: 0.528)
Video: Understanding Aerodynamic Drag
Channel: The Efficient Engineer
Timestamp: 03:55 - 04:39
URL: https://www.youtube.com/watch?v=GMmNKUlXXDs?t=235s
Content: . This is because turbulence introduces a lot of mixing between the different layers of flow, and this momentum transfer means the flow can sustain a larger adverse pressure gradient without separating. This is why golf balls have dimples instead of being perfectly smooth - the dimples generate turb...

[Source 2] (Score: 0.505)
Video: Understanding Laminar and Turbulent Flow
Channel: The Efficient Engineer
Timestamp: 00:39 - 01:26
URL: https://www.youtube.com/watch?v=9A-uUG0WR0w?t=39s
Content: . This is the start of the transition between the laminar and tur

## 9. Create Gradio-Ready Function

This function is ready to be used in the Gradio deployment.

In [29]:
def chat_with_agent(message: str, history: List[List[str]]) -> str:
    """
    Gradio-compatible chat function.
    
    Args:
        message: User's message
        history: Chat history (list of [user_msg, bot_msg] pairs)
    
    Returns:
        Agent's response
    """
    # Clear memory and rebuild from history
    memory.clear()
    
    for user_msg, bot_msg in history:
        memory.chat_memory.add_user_message(user_msg)
        memory.chat_memory.add_ai_message(bot_msg)
    
    # Get response
    try:
        response = agent_executor.invoke({
            "input": message,
            "current_date": datetime.now().strftime("%Y-%m-%d"),
        })
        return response["output"]
    except Exception as e:
        return f"I encountered an error: {str(e)}\n\nPlease try rephrasing your question."


print("✅ Gradio-compatible chat function created")
print("   Function: chat_with_agent(message, history)")
print("   Ready for deployment!")

✅ Gradio-compatible chat function created
   Function: chat_with_agent(message, history)
   Ready for deployment!


## 10. Agent Evaluation with LangSmith

In [30]:
# Create evaluation dataset
eval_questions = [
    "What is stress?",
    "Explain Young's modulus",
    "How do materials fail?",
    "What is the difference between stress and strain?",
    "Which videos discuss fatigue?",
]

print("\n📊 Running Agent Evaluation...\n")

eval_results = []
for i, question in enumerate(eval_questions, 1):
    print(f"[{i}/{len(eval_questions)}] {question}")
    
    try:
        response = ask_agent(question)
        success = len(response) > 20 and "error" not in response.lower()
        
        eval_results.append({
            "question": question,
            "response_length": len(response),
            "success": success,
        })
        
        print(f"   ✅ Response: {len(response)} chars\n")
    
    except Exception as e:
        print(f"   ❌ Error: {e}\n")
        eval_results.append({
            "question": question,
            "response_length": 0,
            "success": False,
        })

# Calculate metrics
success_rate = sum(r["success"] for r in eval_results) / len(eval_results)
avg_response_length = sum(r["response_length"] for r in eval_results) / len(eval_results)

print("\n" + "="*80)
print("📊 EVALUATION RESULTS")
print("="*80)
print(f"Success Rate: {success_rate:.1%}")
print(f"Avg Response Length: {avg_response_length:.0f} chars")
print(f"\n✅ All traces available in LangSmith dashboard!")
print("="*80)


📊 Running Agent Evaluation...

[1/5] What is stress?


> Entering new AgentExecutor chain...



Invoking: `search_transcripts` with `{'query': 'stress engineering'}`



🔍 Tool called with query: 'stress engineering'
[Source 1] (Score: 0.477)
Video: Understanding True Stress and True Strain
Channel: The Efficient Engineer
Timestamp: 02:32 - 03:18
URL: https://www.youtube.com/watch?v=AkX6JqlWRqc?t=152s
Content: . Examples of this might be the analysis of manufacturing processes, or performing finite element analysis which models large strains. By making a few assumptions we can calculate the true stress-strain curve based on the engineering curve, and so we can avoid having to measure the instantaneous cro...

[Source 2] (Score: 0.434)
Video: Why are plane windows round?
Channel: Real Engineering
Timestamp: 00:59 - 01:36
URL: https://www.youtube.com/watch?v=7rXGRPMD-GQ?t=59s
Content: is stretched more and more the stress begins to rise eventually the stress can rise so high that the material breaks there are a lot of factors that can Elevate the stress one of them is the shape of 

c:\Users\yogan\Documents\JupyterNotebook_ironhack\Bootcamp\week_08_03_final_project\Final-Project---Youtube-Engineering-Chatbot\venv\Lib\site-packages\langchain\memory\chat_memory.py:38: UserWarning: 'ConversationBufferMemory' got multiple output keys: dict_keys(['output', 'intermediate_steps']). The default 'output' key is being used. If this is not desired, please manually set 'output_key'.
  warnings.warn(



Invoking: `search_transcripts` with `{'query': "Young's modulus"}`



🔍 Tool called with query: 'Young's modulus'
[Source 1] (Score: 0.731)
Video: Understanding Young's Modulus
Channel: The Efficient Engineer
Timestamp: 05:13 - 06:02
URL: https://www.youtube.com/watch?v=DLE-ieOVFjI?t=313s
Content: . Young's modulus is a crucially important material property when it comes to engineering. In engineering design, a common objective for many different applications is to keep elastic deformations as small as possible, which means that Young's modulus is a key parameter that needs to be considered i...

[Source 2] (Score: 0.698)
Video: Understanding Young's Modulus
Channel: The Efficient Engineer
Timestamp: 01:55 - 02:31
URL: https://www.youtube.com/watch?v=DLE-ieOVFjI?t=115s
Content: . The higher the Young's modulus, the stiffer of material and so the smaller the elastic deformations will be for a given applied load. If we perform tensile tests for a few different materials we will notice t

c:\Users\yogan\Documents\JupyterNotebook_ironhack\Bootcamp\week_08_03_final_project\Final-Project---Youtube-Engineering-Chatbot\venv\Lib\site-packages\langchain\memory\chat_memory.py:38: UserWarning: 'ConversationBufferMemory' got multiple output keys: dict_keys(['output', 'intermediate_steps']). The default 'output' key is being used. If this is not desired, please manually set 'output_key'.
  warnings.warn(


...

c:\Users\yogan\Documents\JupyterNotebook_ironhack\Bootcamp\week_08_03_final_project\Final-Project---Youtube-Engineering-Chatbot\venv\Lib\site-packages\langchain\memory\chat_memory.py:38: UserWarning: 'ConversationBufferMemory' got multiple output keys: dict_keys(['output', 'intermediate_steps']). The default 'output' key is being used. If this is not desired, please manually set 'output_key'.
  warnings.warn(



Invoking: `search_transcripts` with `{'query': 'difference between stress and strain'}`



🔍 Tool called with query: 'difference between stress and strain'
[Source 1] (Score: 0.611)
Video: Understanding True Stress and True Strain
Channel: The Efficient Engineer
Timestamp: 04:54 - 05:24
URL: https://www.youtube.com/watch?v=AkX6JqlWRqc?t=294s
Content: . Because of the form of this equation, true strain is also known as logarithmic strain, or natural strain. I hope this has helped explain the differences between engineering and true stress-strain curves. Thanks for watching, and remember to hit the subscribe button and the bell to be notified abou...

[Source 2] (Score: 0.601)
Video: An Introduction to Stress and Strain
Channel: The Efficient Engineer
Timestamp: 00:00 - 00:52
URL: https://www.youtube.com/watch?v=aQf6Q8t1FQE?t=0s
Content: Stress and strain are fundamental concepts that are used to describe how a body responds to external loads. In this video we'll explore these concepts 

c:\Users\yogan\Documents\JupyterNotebook_ironhack\Bootcamp\week_08_03_final_project\Final-Project---Youtube-Engineering-Chatbot\venv\Lib\site-packages\langchain\memory\chat_memory.py:38: UserWarning: 'ConversationBufferMemory' got multiple output keys: dict_keys(['output', 'intermediate_steps']). The default 'output' key is being used. If this is not desired, please manually set 'output_key'.
  warnings.warn(



Invoking: `find_videos` with `{'topic': 'fatigue'}`


Found 2 videos about 'fatigue':

1. Understanding Fatigue Failure and S-N Curves
   Channel: The Efficient Engineer
   URL: https://www.youtube.com/watch?v=o-6V_JoRX1g
2. The Material Science of Metal 3D Printing
   Channel: Real Engineering
   URL: https://www.youtube.com/watch?v=fzBRYsiyxjIHere are some videos that discuss fatigue:

1. **Understanding Fatigue Failure and S-N Curves**
   - Channel: The Efficient Engineer
   - [Watch here](https://www.youtube.com/watch?v=o-6V_JoRX1g)

2. **The Material Science of Metal 3D Printing**
   - Channel: Real Engineering
   - [Watch here](https://www.youtube.com/watch?v=fzBRYsiyxjI)

> Finished chain.
   ✅ Response: 339 chars


📊 EVALUATION RESULTS
Success Rate: 100.0%
Avg Response Length: 1117 chars

✅ All traces available in LangSmith dashboard!


c:\Users\yogan\Documents\JupyterNotebook_ironhack\Bootcamp\week_08_03_final_project\Final-Project---Youtube-Engineering-Chatbot\venv\Lib\site-packages\langchain\memory\chat_memory.py:38: UserWarning: 'ConversationBufferMemory' got multiple output keys: dict_keys(['output', 'intermediate_steps']). The default 'output' key is being used. If this is not desired, please manually set 'output_key'.
  warnings.warn(


## 11. Export Agent for Deployment

In [31]:
# Save agent configuration
agent_config = {
    "created_at": datetime.now().isoformat(),
    "model": CHAT_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "index_name": INDEX_NAME,
    "namespace": NAMESPACE,
    "num_tools": len(tools),
    "tool_names": [tool.name for tool in tools],
    "memory_enabled": True,
    "langsmith_project": os.environ["LANGCHAIN_PROJECT"],
    "evaluation_success_rate": f"{success_rate:.1%}",
}

config_path = "../config/agent_config.json"
os.makedirs(os.path.dirname(config_path), exist_ok=True)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(agent_config, f, indent=2)

print(f"\n💾 Agent configuration saved to: {config_path}")
print(json.dumps(agent_config, indent=2))


💾 Agent configuration saved to: ../config/agent_config.json
{
  "created_at": "2026-03-09T17:06:47.698732",
  "model": "gpt-4o",
  "embedding_model": "text-embedding-3-small",
  "index_name": "youtube-rag-mechanical-engineering",
  "namespace": "efficient-engineer-v3",
  "num_tools": 3,
  "tool_names": [
    "search_transcripts",
    "find_videos",
    "get_video_info"
  ],
  "memory_enabled": true,
  "langsmith_project": "youtube-rag-engineering-chatbot",
  "evaluation_success_rate": "100.0%"
}
